# PS1 C17 Final Kaggle Notebook

In [ ]:
import os
import gc
import time
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, classification_report, confusion_matrix, roc_auc_score, top_k_accuracy_score, roc_curve, auc
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False

seed_everything(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

In [ ]:
DATASET_ROOT = Path("/kaggle/input/datasets/mohdshad/17-fracture-type-classes/Human Bone Fracture C17 Dataset")
OUTPUT_DIR = Path("/kaggle/working/ps1_c17_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 300
BATCH_SIZE = 16
EPOCHS = 18
LR = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
PATIENCE = 6
LABEL_SMOOTHING = 0.08

print(DATASET_ROOT.exists())
print(sorted([p.name for p in DATASET_ROOT.iterdir() if p.is_dir()]))

In [ ]:
def scan_dataset(root):
    root = Path(root)
    rows = []
    valid_ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    for class_dir in sorted(root.iterdir()):
        if not class_dir.is_dir():
            continue
        label_name = class_dir.name
        for img_path in class_dir.rglob("*"):
            if img_path.is_file() and img_path.suffix.lower() in valid_ext:
                rows.append({"path": str(img_path), "label_name": label_name})
    return pd.DataFrame(rows)

raw_df = scan_dataset(DATASET_ROOT)

bad_files = []
for p in raw_df["path"]:
    try:
        with Image.open(p) as img:
            img.verify()
    except Exception as e:
        bad_files.append((p, str(e)))

if bad_files:
    bad_paths = set(p for p, _ in bad_files)
    raw_df = raw_df[~raw_df["path"].isin(bad_paths)].reset_index(drop=True)

class_names = sorted(raw_df["label_name"].unique())
class_to_idx = {cls: i for i, cls in enumerate(class_names)}
idx_to_class = {i: cls for cls, i in class_to_idx.items()}
raw_df["label"] = raw_df["label_name"].map(class_to_idx)

print("Total images:", len(raw_df))
print("Num classes:", len(class_names))
display(raw_df["label_name"].value_counts().rename_axis("class").reset_index(name="count"))

In [ ]:
train_df, temp_df = train_test_split(
    raw_df,
    test_size=0.30,
    stratify=raw_df["label_name"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label_name"],
    random_state=42
)

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(df))
    print(df["label_name"].value_counts().sort_index())
    print("-" * 60)

In [ ]:
class FractureDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        label = int(row["label"])
        return image, label, row["path"]

train_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(12),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.92, 1.08)),
    transforms.ColorJitter(brightness=0.18, contrast=0.18),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.22, scale=(0.02, 0.10), ratio=(0.3, 3.3))
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = FractureDataset(train_df, transform=train_tfms)
val_ds = FractureDataset(val_df, transform=eval_tfms)
test_ds = FractureDataset(test_df, transform=eval_tfms)

class_counts = train_df["label"].value_counts().sort_index().to_numpy()
class_weights = 1.0 / class_counts
sample_weights = train_df["label"].map(lambda x: class_weights[x]).to_numpy()
sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

images, labels, paths = next(iter(train_loader))
print(images.shape, labels.shape)
print(class_names[labels[0].item()], paths[0])

In [ ]:
plt.figure(figsize=(14, 8))
for i in range(8):
    ax = plt.subplot(2, 4, i + 1)
    img = images[i].permute(1, 2, 0).cpu().numpy()
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(class_names[labels[i].item()], fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
weights = models.EfficientNet_B3_Weights.DEFAULT
model = models.efficientnet_b3(weights=weights)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, len(class_names))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE),
    label_smoothing=LABEL_SMOOTHING
)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.05)

num_params = sum(p.numel() for p in model.parameters()) / 1e6
print("Parameters (M):", round(num_params, 2))

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss = 0.0
    all_preds = []
    all_targets = []

    for images, labels, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, labels)

        if train_mode:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_targets.extend(labels.detach().cpu().numpy().tolist())

    epoch_loss = total_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    epoch_f1 = f1_score(all_targets, all_preds, average="macro")
    return epoch_loss, epoch_acc, epoch_f1

best_path = OUTPUT_DIR / "best_model.pth"
history = []
best_f1 = -1.0
best_epoch = -1
patience_left = PATIENCE

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    train_loss, train_acc, train_f1 = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, val_f1 = run_epoch(model, val_loader, criterion, optimizer=None)
    lr_now = optimizer.param_groups[0]["lr"]
    scheduler.step()

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_accuracy": train_acc,
        "val_accuracy": val_acc,
        "train_macro_f1": train_f1,
        "val_macro_f1": val_f1,
        "overfitting_gap": val_loss - train_loss,
        "learning_rate": lr_now,
        "epoch_seconds": time.time() - start
    }
    history.append(row)

    improved = val_f1 > best_f1
    if improved:
        best_f1 = val_f1
        best_epoch = epoch
        patience_left = PATIENCE
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "image_size": IMAGE_SIZE,
            "epoch": epoch,
            "best_val_f1": best_f1
        }, best_path)
    else:
        patience_left -= 1

    print(
        f"Epoch {epoch:02d}/{EPOCHS}  "
        f"TrLoss={train_loss:.4f}  TrAcc={train_acc:.4f}  TrF1={train_f1:.4f}  "
        f"VaLoss={val_loss:.4f}  VaAcc={val_acc:.4f}  VaF1={val_f1:.4f}  "
        f"LR={lr_now:.6f}" + ("  ★" if improved else "")
    )

    if patience_left <= 0:
        print("Early stopping")
        break

history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "history.csv", index=False)

summary_rows = pd.DataFrame([
    {"epoch": "SUMMARY", "train_loss": None, "val_loss": None, "train_accuracy": None, "val_accuracy": None, "train_macro_f1": None, "val_macro_f1": None, "overfitting_gap": history_df["overfitting_gap"].max(), "learning_rate": None, "epoch_seconds": history_df["epoch_seconds"].sum()},
    {"epoch": "BEST_EPOCH", "train_loss": None, "val_loss": None, "train_accuracy": None, "val_accuracy": None, "train_macro_f1": None, "val_macro_f1": best_f1, "overfitting_gap": None, "learning_rate": None, "epoch_seconds": best_epoch}
])

model_perf_df = pd.concat([history_df, summary_rows], ignore_index=True)
model_perf_df.to_csv(OUTPUT_DIR / "model_performance_analysis.csv", index=False)

print("Best epoch:", best_epoch, "Best val macro F1:", round(best_f1, 4))

In [ ]:
plt.figure(figsize=(18, 6))

plt.subplot(1, 2, 1)
plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_df["epoch"], history_df["train_accuracy"], label="train_acc")
plt.plot(history_df["epoch"], history_df["val_accuracy"], label="val_acc")
plt.plot(history_df["epoch"], history_df["val_macro_f1"], label="val_macro_f1")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Accuracy / F1")
plt.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
checkpoint = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

def predict_loader(model, loader):
    all_targets = []
    all_preds = []
    all_probs = []
    all_paths = []
    times = []

    with torch.no_grad():
        for images, labels, paths in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            start = time.time()
            logits = model(images)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            times.append(time.time() - start)

            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_targets.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
            all_paths.extend(list(paths))

    infer_ms = (sum(times) / len(loader.dataset)) * 1000 if len(loader.dataset) > 0 else 0.0
    return np.array(all_targets), np.array(all_preds), np.array(all_probs), all_paths, infer_ms

val_targets, val_preds, val_probs, val_paths, val_ms = predict_loader(model, val_loader)
test_targets, test_preds, test_probs, test_paths, test_ms = predict_loader(model, test_loader)

print("Validation Accuracy:", round(accuracy_score(val_targets, val_preds), 4))
print("Validation Macro F1:", round(f1_score(val_targets, val_preds, average="macro"), 4))
print("Test Accuracy:", round(accuracy_score(test_targets, test_preds), 4))
print("Test Macro F1:", round(f1_score(test_targets, test_preds, average="macro"), 4))
print("Inference ms/image:", round(test_ms, 2))

In [ ]:
test_precision_macro, test_recall_macro, test_f1_macro, _ = precision_recall_fscore_support(test_targets, test_preds, average="macro")
test_precision_weighted, test_recall_weighted, test_f1_weighted, _ = precision_recall_fscore_support(test_targets, test_preds, average="weighted")
test_acc = accuracy_score(test_targets, test_preds)
test_top3 = top_k_accuracy_score(test_targets, test_probs, k=min(3, len(class_names)), labels=np.arange(len(class_names)))

y_test_bin = label_binarize(test_targets, classes=np.arange(len(class_names)))
roc_auc_macro = roc_auc_score(y_test_bin, test_probs, multi_class="ovr", average="macro")
roc_auc_weighted = roc_auc_score(y_test_bin, test_probs, multi_class="ovr", average="weighted")

report = classification_report(test_targets, test_preds, target_names=class_names, output_dict=True, zero_division=0)
per_class_rows = []
for cls in class_names:
    row = report[cls]
    cls_idx = class_to_idx[cls]
    auc_cls = roc_auc_score((test_targets == cls_idx).astype(int), test_probs[:, cls_idx])
    per_class_rows.append({
        "class_name": cls,
        "precision": row["precision"],
        "recall": row["recall"],
        "f1_score": row["f1-score"],
        "support": int(row["support"]),
        "auc_ovr": auc_cls
    })

per_class_df = pd.DataFrame(per_class_rows).sort_values("class_name")
per_class_df.to_csv(OUTPUT_DIR / "per_class_metrics.csv", index=False)

final_results_df = pd.DataFrame([
    {"metric_name": "Accuracy", "overall_value": test_acc, "interpretation": "Overall correctness"},
    {"metric_name": "Macro Precision", "overall_value": test_precision_macro, "interpretation": "Unweighted class precision"},
    {"metric_name": "Macro Recall", "overall_value": test_recall_macro, "interpretation": "Unweighted class recall"},
    {"metric_name": "Macro F1", "overall_value": test_f1_macro, "interpretation": "Balanced class-wise performance"},
    {"metric_name": "Weighted F1", "overall_value": test_f1_weighted, "interpretation": "Support-weighted F1"},
    {"metric_name": "Top-3 Accuracy", "overall_value": test_top3, "interpretation": "Whether true class appears in top 3"},
    {"metric_name": "ROC-AUC OVR Macro", "overall_value": roc_auc_macro, "interpretation": "One-vs-rest multiclass AUC"},
    {"metric_name": "ROC-AUC OVR Weighted", "overall_value": roc_auc_weighted, "interpretation": "Support-weighted one-vs-rest multiclass AUC"},
    {"metric_name": "Inference Time (ms/image)", "overall_value": test_ms, "interpretation": "Latency per image"},
    {"metric_name": "Parameters (Millions)", "overall_value": num_params, "interpretation": "Model size in millions of parameters"}
])

final_results_df.to_csv(OUTPUT_DIR / "final_results.csv", index=False)

test_pred_df = pd.DataFrame({
    "path": test_paths,
    "true_idx": test_targets,
    "pred_idx": test_preds,
    "true_label": [idx_to_class[x] for x in test_targets],
    "pred_label": [idx_to_class[x] for x in test_preds],
    "confidence": test_probs.max(axis=1)
})
for i, cls in enumerate(class_names):
    test_pred_df[f"prob_{cls}"] = test_probs[:, i]
test_pred_df.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

display(final_results_df)
display(per_class_df.head(17))

In [ ]:
cm = confusion_matrix(test_targets, test_preds)
fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm, cmap="viridis")
ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_xticks(np.arange(len(class_names)))
ax.set_yticks(np.arange(len(class_names)))
ax.set_xticklabels(class_names, rotation=60, ha="right", fontsize=9)
ax.set_yticklabels(class_names, fontsize=9)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() * 0.5 else "black", fontsize=8)

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
fpr_micro, tpr_micro, _ = roc_curve(y_test_bin.ravel(), test_probs.ravel())
auc_micro = auc(fpr_micro, tpr_micro)

plt.figure(figsize=(12, 9))
plt.plot(fpr_micro, tpr_micro, lw=2.5, label=f"micro-average (AUC={auc_micro:.3f})")
for i, cls in enumerate(class_names):
    fpr_i, tpr_i, _ = roc_curve(y_test_bin[:, i], test_probs[:, i])
    auc_i = auc(fpr_i, tpr_i)
    plt.plot(fpr_i, tpr_i, alpha=0.35, lw=1.2, label=f"{cls} ({auc_i:.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curves")
plt.legend(loc="lower right", fontsize=8, ncol=1)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curve.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.forward_handle = self.target_layer.register_forward_hook(self.save_activation)
        self.backward_handle = self.target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, inp, out):
        self.activations = out.detach()

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.zero_grad(set_to_none=True)
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = int(output.argmax(dim=1).item())
        score = output[:, class_idx].sum()
        score.backward(retain_graph=True)

        grads = self.gradients[0]
        acts = self.activations[0]
        weights = grads.mean(dim=(1, 2), keepdim=True)
        cam = (weights * acts).sum(dim=0)
        cam = torch.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        cam = cam.cpu().numpy()
        return cam

    def close(self):
        self.forward_handle.remove()
        self.backward_handle.remove()

def denorm_tensor(x):
    mean = torch.tensor([0.485, 0.456, 0.406], device=x.device).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=x.device).view(3, 1, 1)
    x = x * std + mean
    return x.clamp(0, 1)

target_layer = model.features[-1]
gradcam = GradCAM(model, target_layer)

correct_indices = np.where(test_targets == test_preds)[0]
selected_indices = correct_indices[:6] if len(correct_indices) >= 6 else np.arange(min(6, len(test_ds)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, idx in zip(axes, selected_indices):
    img, label, path = test_ds[idx]
    input_tensor = img.unsqueeze(0).to(DEVICE)
    pred = model(input_tensor).argmax(dim=1).item()
    cam = gradcam.generate(input_tensor, pred)

    img_show = denorm_tensor(input_tensor[0]).permute(1, 2, 0).detach().cpu().numpy()
    heatmap = Image.fromarray(np.uint8(cam * 255)).resize((img_show.shape[1], img_show.shape[0]))
    heatmap = np.array(heatmap) / 255.0

    ax.imshow(img_show)
    ax.imshow(heatmap, cmap="jet", alpha=0.42)
    ax.set_title(f"True: {idx_to_class[label]}\nPred: {idx_to_class[pred]}", fontsize=10)
    ax.axis("off")

for j in range(len(selected_indices), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "gradcam_examples.png", dpi=180, bbox_inches="tight")
plt.show()

gradcam.close()

In [ ]:
readme = f"""
Team Name: Perceptra
Dataset: Human Bone Fracture C17 Dataset
Classes: {len(class_names)}
Model: efficientnet_b3
Image Size: {IMAGE_SIZE}
Epochs Run: {len(history_df)}
Best Epoch: {best_epoch}
Best Validation Macro F1: {best_f1:.4f}
Test Accuracy: {test_acc:.4f}
Test Macro F1: {test_f1_macro:.4f}
ROC-AUC OVR Macro: {roc_auc_macro:.4f}
Inference Time (ms/image): {test_ms:.2f}

Files:
best_model.pth
final_results.csv
model_performance_analysis.csv
per_class_metrics.csv
test_predictions.csv
training_curves.png
confusion_matrix.png
roc_curve.png
gradcam_examples.png
"""

with open(OUTPUT_DIR / "README.txt", "w") as f:
    f.write(readme)

with open(OUTPUT_DIR / "TEAM.txt", "w") as f:
    f.write("Team Name: Perceptra\nTeam Lead: SHAD\nEmail: 23f3004148@ds.study.iitm.ac.in\n")

print(readme)
print(sorted([p.name for p in OUTPUT_DIR.iterdir()]))